In [1]:
import pandas as pd
import altair as alt

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Dataset shape: (4747, 29)
Columns: ['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [2]:
season_order = ['Winter', 'Spring', 'Summer', 'Fall']

df1 = df[['season', 'classification']].dropna().copy()
df1 = df1[df1['season'].isin(season_order)]

chart1 = (
    alt.Chart(df1)
    .mark_rect()
    .encode(
        x=alt.X('season:N', sort=season_order, title='Season'),
        y=alt.Y('classification:N', title='Report Classification'),
        color=alt.Color(
            'count():Q',
            scale=alt.Scale(scheme='viridis'),
            title='Number of Reports'
        ),
        tooltip=[
            alt.Tooltip('season:N', title='Season'),
            alt.Tooltip('classification:N', title='Classification'),
            alt.Tooltip('count():Q', title='Number of Reports')
        ]
    )
    .properties(
        title='Bigfoot Sightings by Season and Classification',
        width=550,
        height=300
    )
)

chart1


alt.Chart(...)

In [3]:
df2 = df[['date', 'season', 'classification']].copy()

df2['date'] = pd.to_datetime(
    df2['date'], errors='coerce', format='mixed'
)

df2 = df2.dropna()

df2['year'] = df2['date'].dt.year

df2 = df2[
    (df2['year'].between(1990, 2022)) &
    (df2['season'].isin(season_order)) &
    (df2['classification'].isin(['Class A', 'Class B']))
]

# Interactive season dropdown
season_choice = alt.param(
    name='season_choice',
    value='Summer',
    bind=alt.binding_select(
        options=['Winter', 'Spring', 'Summer', 'Fall'],
        name='Select Season: '
    )
)

chart2 = (
    alt.Chart(df2)
    .transform_filter(alt.datum.season == season_choice)
    .mark_line(point=True)
    .encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('count():Q', title='Number of Reports'),
        color=alt.Color(
            'classification:N',
            title='Report Classification'
        ),
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('classification:N', title='Classification'),
            alt.Tooltip('count():Q', title='Number of Reports')
        ]
    )
    .add_params(season_choice)
    .properties(
        title='Bigfoot Sightings by Year and Classification',
        width=650,
        height=350
    )
)

chart2


alt.Chart(...)

In [4]:
from pathlib import Path

output_dir = Path.cwd()

chart1.save(str(output_dir / "hw5_bigfoot_chart1.json"))
chart2.save(str(output_dir / "hw5_bigfoot_chart2.json"))

print("Chart 1 saved:", (output_dir / "hw5_bigfoot_chart1.json").exists())
print("Chart 2 saved:", (output_dir / "hw5_bigfoot_chart2.json").exists())

Chart 1 saved: True
Chart 2 saved: True
